In [ ]:
import ast
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import yaml

with open("panel_sizes_cm.yaml", "r") as file:
    panel_sizes = yaml.safe_load(file)

In [ ]:
folder = "../../../main_results/LLM-free_example"
run_id = "35e85ba1-0b04-43e8-b95d-cd57e275ef26"

In [ ]:
# Load the full experimental results
summary = pd.read_csv(os.path.join(folder, "summary.csv"))

# Select the entries belonging to the specified run id
summary = summary[summary["run_id"] == run_id]

# Get the ess
ess = summary.groupby("iteration")["ess"].unique().values
ess = [c[0] for c in ess]

config_str = summary["config"].unique()[0]
config = ast.literal_eval(config_str)

threshold = config["method"]["ess_threshold"] * config["method"]["particle_pool_size"]

In [ ]:
f = "../../"
style_file = os.path.join(f, ".matplotlibrc")

with mpl.rc_context(fname=style_file):
    fig, ax = plt.subplots(
        figsize=(
            panel_sizes["panel_trajectory"]["width_cm"] / 2.54 * 1.0135,
            panel_sizes["panel_trajectory"]["height_cm"] / 2.54,
        )
    )

    ax.hlines(threshold, 0, len(ess), color="k", lw=1.0, label=r"$\tau_{ESS}$")

    # Plot the ESS and ignore the initial entry, i.e. shift it by one compared to the
    # ratios. This is motivated by the way the ESS is stored: For the pool after
    # iteration i, the ESS is computed in the resampling step of iteration i+1, so there
    # is a one step lag in the recording of the ESS compared to the ratios of the
    # correct model.
    ax.plot(
        ess[1:],
        c="darkblue",
        lw=2,
        label="ESS",
        marker="o",
        markersize=4.0,
    )

    ax.set_xlabel("iteration")
    ax.set_ylabel("ESS", labelpad=10.3)
    ax.set_ylim(bottom=-10.0)
    ax.set_xlim(right=21, left=-1)
    ax.set_yticks([0.0, 50.0], ["0", "50"])
    ax.legend(loc="lower right")

    fig.tight_layout()

    save_path = f"../panels/ess_{run_id}.svg"
    fig.savefig(save_path, bbox_inches="tight", format="svg", transparent=True)
    print(f"Figure saved to {save_path}")
    plt.close(fig)